# 策略概述

## 一句話說明

一般的 Z-Score 配對交易，進出場門檻是**固定**的（例如價差偏離 2 個標準差時進場、回到 0 時出場）。本策略改為讓一個**會從歷史經驗學習的模型**，依據每一組配對在觀察期（形成期）展現的性質（回歸快不快、波動大不大、訊號多不多……），**為每一組配對量身挑選最適合它的進出場門檻**，甚至判斷「這組配對這一期不值得做，直接空手跳過」。

> **比喻**：固定門檻像是所有配對都穿同一個尺寸的衣服；本策略則依每組配對的「體型」量身選門檻。挑選的依據，是從歷史上「同類配對改用不同門檻分別會賺或賠多少」累積出來的經驗。

**與純 Z-Score 的唯一差別是「門檻怎麼定」**——價差（spread）計算、下單規則、停損、費用全部相同。因此兩者比較時，唯一的變因就是「門檻是**固定** vs 由模型**學習挑選**」，這讓實驗成為乾淨的單變因對照。


# 輸入值卡

::: {.callout-important}

### 這個模型吃什麼、吐什麼

| 項目 | 內容 |
| :--- | :--- |
| **輸入特徵** | **12 維**，全部由**形成期**視窗算出（交易期開始前即已確定，無前視）。各自縮放並截尾至約 $[-3,3]$，明細見階段 2 |
| **輸出** | 9 個動作的預測報酬（%），決策取 $\arg\max$ |
| **動作選單** | SKIP ＋ 8 組 $(entry\_z, exit\_z) \in \{1.5, 2.0, 2.5, 3.0\} \times \{0.0, 0.5\}$ |
| **網路** | MLP 12 → 64 → 64 → 9（ReLU），Adam，學習率 $10^{-3}$ |
| **訓練標籤** | **9 個動作的報酬全部反事實回算**（逐一以 Z-Score 狀態機模擬同一段交易期）。SKIP 恆為 0 |
| **損失** | 全 9 維 MSE，每期 40 epoch |
| **訓練資料切分** | walk-forward：期 $k$ 只用「交易期已於 $k$ 開始前結束」的樣本（$t_e < trade\_start_k$） |
| **啟用學習門檻** | 可用樣本 $\ge 200$；未達時退回基準動作 $(2.0, 0.0)$ → 暖身期行為 $\equiv$ Z-Score |
| **隨機性** | ⚠ **不固定種子**（權重初始化 + 批次洗牌）。單次回測的數字是隨機變數，一律以**五輪獨立重跑的中位數與全距**報告 |

:::

::: {.callout-warning}

### 這不是強化學習——名稱裡的 DRL 只是資料庫代號

9 個動作的報酬事後**全部可精確回算**，模型等於拿到完整答案卷。依 Sutton & Barto 的
框架，探索之所以必要是因為「未執行的動作觀察不到報酬」；本問題不具此性質，
於是退化為**全資訊監督回歸**——無探索、無價值迭代、無序列信用分配。

`result.db` 的 `METHOD` 欄位沿用歷史字串 `Grid (AGG-SSD-DRL)`、`SSD (DRL)`，
其中 `DRL` 指的就是本策略；為求結果可追溯不改寫欄位值，但敘述一律用 **DL-THR**。

真正的部分回饋版本見 `trading/rl_threshold_trading.ipynb`（**RL-THR**）。

:::

# 策略架構

交易期對每組配對、每一期做一次決策：先算特徵，再由模型挑一個進出場門檻（或跳過），選定後就用固定規則執行——**與 Z-Score 基準的唯一差別，是這個門檻由模型學來、而非人工固定**。

```{mermaid}
flowchart LR
  A["量身特徵<br/>12 個形成期性質"] --> B["模型挑門檻<br/>9 選 1（或跳過）"]
  B --> C["執行<br/>用選定門檻跑 Z-Score 狀態機"]
  C --> D["損益"]
  D -.學習教材.-> E["反事實標籤<br/>滾動前向、無前視"]
  E -.回饋.-> B
```

## 運作流程（每組配對、每一期各做一次）

1. **量身特徵**：從形成期算出 12 個描述這組配對性質的數字（見階段 2）。
2. **模型挑門檻**：一個小型神經網路讀入這 12 個數字，預測 9 個候選門檻各自的期望報酬，挑「預期報酬最高」的那一個。
3. **只用過去、不用未來**：模型只拿「結果在本期開始前就已經完全確定」的歷史配對來學習，杜絕用到未來資訊（前視偏誤）。
4. **從所有選項學習**：對每一組歷史配對，把 9 個門檻**逐一實際模擬一遍**、算出各自的真實報酬當作教材。因為每個選項的結果事後都能精確算出，這其實是單純的「看完整例子學規律」問題（監督式學習），不需要一般 AI 常見的試誤探索。
5. **保底機制**：當可用的歷史例子還不夠（少於 200 筆）時，自動退回固定門檻 $(2.0, 0.0)$，此時行為與純 Z-Score 完全相同——所以本策略的表現「至少不會比 Z-Score 差太多」有結構上的保證。

**動作選單（9 選 1）**：不交易（SKIP）＋ 8 種門檻組合＝進場門檻 $\in\{1.5, 2.0, 2.5, 3.0\}$ × 出場門檻 $\in\{0.0, 0.5\}$。

使用本引擎的策略：SSD Rolling（門檻學習版）、Agglomerative Fundamentals（門檻學習版），各自借用對應 Z-Score 策略的形成期配對。


# 參考文獻與引用對應


## 文獻 1：Kim & Kim (2019)

> Kim, T., & Kim, H. Y. (2019). Optimizing the pairs-trading strategy using deep reinforcement learning with trading and stop-loss boundaries. *Complexity*, 2019, Article 3582516.

**參考部分**：

- 核心框架：讓學習器**輸出交易邊界（進場／出場門檻）而非逐日持倉動作**——
  agent 的動作空間是門檻組合的離散集合，實際交易由既定規則依門檻執行
- 以配對的歷史特徵作為狀態輸入，門檻選擇以期為單位（而非以日為單位）

**為何參考**：

- 本引擎的動作空間設計（8 組 $(entry_z, exit_z)$ + SKIP）直接採用此文獻的門檻選擇式框架：
  把決策粒度從「每日」壓縮到「每配對每期一次」，
  大幅降低動作空間維度，避免模型對日級價格噪音做擇時
- SKIP 動作是對其框架的自然延伸：允許學習器拒絕不值得交易的配對



## 文獻 2：Sutton & Barto (2018)

> Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.

**參考部分**：

- 序貫決策問題的 MDP 形式化 $(S, A, P, R, \gamma)$，以及**探索－利用權衡**的成立條件：
  只有在「未執行的動作觀察不到報酬」（部分回饋）時才需要探索

**為何參考**：

- 用於界定本問題的性質：歷史配對期上**全部 9 個動作的報酬都能精確反事實回算**
  （對交易期價格逐一模擬 9 組門檻），屬於**全資訊回饋**——
  依此理論框架，問題退化為監督回歸，無需探索機制、無需價值迭代，
  這是本引擎採用「反事實標籤 + MSE 回歸」而非 Q-learning 的理論依據

> **白話**：強化學習需要「試誤探索」是因為通常只能看到自己選的那個動作的結果。但本問題裡，每組歷史配對的 9 個門檻報酬事後都能精確算出（全資訊），等於拿到完整答案卷，因此退化成單純的「看例子學規律」（監督回歸），不需要探索、也不需要價值迭代。


## 文獻 3：統計套利機器學習方法綜述 (2025)

> A survey of statistical arbitrage pair trading with machine learning, deep learning, and reinforcement learning methods (2025).

**參考部分**：

- RL 應用於配對交易的動作空間設計分類：逐日持倉決策 vs 門檻／邊界決策兩大路線，
  及其在樣本效率與過度交易風險上的取捨

**為何參考**：

- 提供門檻選擇式路線在文獻中的定位；本引擎的設計取捨（決策粒度、費用敏感性）與其歸納一致



# 各階段行為

引擎對每個交易期的每組配對依序執行：特徵萃取 → 訓練 → 決策 → 反事實標籤生成 → 正式模擬。


## 階段 1：Spread 與 Z-Score（與 Z-Score 狀態機同口徑）

價格先做同樣的異常清理（單日 |漲跌幅| > 50% 以前值遞補），
Z-Score 以形成期凍結參數計算（路徑 B，標準化空間）：

$$Z_t = \text{clip}\left(\frac{P'_{A,t} - \beta P'_{B,t} - \mu_\epsilon^{form}}{\max(\sigma_\epsilon^{form}, 10^{-8})},\ -10,\ 10\right), \qquad
P'_{i,t} = \frac{\ln P_{i,t} - \mu_i^{form}}{\sigma_i^{form}}$$

spread 定義與 Z-Score 基準完全相同——確保門檻選擇是唯一的差異來源。


## 階段 2：12 維形成期特徵萃取

對**形成期**（非交易期）的 Z 序列與對數價格計算 12 個特徵，全部**標準化並截尾至約 $[-3, 3]$**。

::: {.callout-note}

**為何要截尾至約 $[-3, 3]$**：這 12 個特徵原本的單位與量級差異很大——例如相關係數落在 $0\sim1$、半衰期可能高達上百天、Z 值偶爾瞬間飆到 $\pm10$。若把這些量級懸殊的數字直接餵給神經網路，模型會被少數大數值主導、也容易被極端離群值干擾而學不穩定。因此每個特徵都先**縮放到共同尺度、並把超出範圍的極端值截平**（clip）：

- **共同尺度**讓每個輸入的「重要性起點」一致，模型才不會單憑量級大小誤判特徵重要性；
- **截尾**把罕見的極端值壓回合理範圍，避免單一離群樣本扭曲訓練。

選 $[-3, 3]$ 是因為若特徵近似常態分布，$\pm3$ 標準差已涵蓋約 99.7% 的正常情形，超出者合理視為離群值而截平。這是機器學習標準的**特徵前處理（feature scaling）**，與金融上把不同指標標準化到可比較尺度的作法同理。

**這個範圍會擷取到怎樣的值**：縮放後，$0$ 附近代表「接近該配對的典型/平均狀態」，$\pm1$ 左右是輕微偏離，接近 $\pm3$ 已屬相當極端但仍完整保留其方向與大小；真正罕見的極端值（例如 Z 值瞬間衝到 $\pm10$、半衰期估到上百天）則一律壓平到邊界 $\pm3$——模型仍知道「這是極端情形」，但不會被單一離群值的絕對大小綁架。

:::

| # | 特徵 | 計算 | 捕捉的資訊 |
| :---: | :--- | :--- | :--- |
| 1 | 期末 $Z$ | $\text{clip}(z_{-1}/3)$ | 進場初始偏離方向 |
| 2 | 期末 $|Z|$ | $\text{clip}(|z_{-1}|/3)$ | 偏離幅度 |
| 3 | 零穿越頻率 | $\text{clip}(zc \times 10)$ | 回歸訊號密度 |
| 4 | 半衰期 | $\ln(HL)/3$（AR(1) 估計，截尾 $[1, 252]$） | 回歸速度 |
| 5 | 近期波動 regime | 近 21 日 $\sigma_z$ ／全期 $\sigma_z - 1$ | 波動狀態變化 |
| 6 | 近期 $Z$ 趨勢 | $(z_{-1} - \bar{z}_{21})/3$ | 短期方向 |
| 7 | 兩股相關係數 | $\text{corr}(\ln P_A, \ln P_B)$ | 共動強度 |
| 8 | 波動率比 | $\sigma_A/\sigma_B - 1$ | 兩腳對稱性 |
| 9 | 對沖比例偏移 | $\beta - 1$ | 曝險不對稱 |
| 10 | Spread 振幅 | $\sigma_\epsilon \times 5$ | 費用可行性（振幅需覆蓋摩擦成本） |
| 11 | $|Z|>2$ 佔比 | $\text{mean}(|z| > 2)$ | 訊號出現頻率 |
| 12 | 最大 $|Z|$ | $\max|z| / 5$ | 極端偏離歷史 |

每個特徵除以 3 或 5、再截平，都是為了達到上述「共同尺度 + 截離群」的目的。形成期資料不足 60 日時無法穩定建構特徵 → 該配對直接使用基準門檻。


## 階段 3：滾動前向訓練（無前視保證）

**滾動前向（walk-forward）**：站在每一期的當下，只用「那個時間點之前就已經完全結束、報酬已成定局」的配對當教材，不能偷看之後才發生的事——這是量化金融杜絕前視偏誤的標準做法。

**樣本資格**：訓練資料庫（緩衝區）中每筆樣本都記錄它的交易期結束日 $t_e$；本期（交易起始日 $trade\_start_k$）只用「已經完結」的樣本訓練：

$$\text{可用樣本} = \{(f, r, t_e) \in \text{緩衝區} : t_e < trade\_start_k\}$$

**決策網路（一個小型神經網路）**：輸入 12 個特徵 → 兩層各 64 個神經元（中間接 ReLU，一種把負值歸零、正值保留的非線性轉換，讓網路能表達較複雜的關係）→ 輸出 9 個數字（對應 9 個門檻的預測報酬）。

**訓練方式**：以「預測報酬與實際報酬的差距平方之平均」（均方誤差）為要縮小的目標，用一種會自動調整步長的梯度下降法（Adam，學習率 $10^{-3}$）反覆調整網路參數；每輪把資料完整看過一遍（稱為一個 epoch，共 40 輪），每次隨機抽一小批樣本更新。只有在可用樣本數 $\ge 200$、且較上次訓練有新樣本時才重新訓練。

**數值變異性**：網路的初始參數與抽樣順序不固定隨機種子，故單次回測的數字是隨機變數；因此以多次獨立重跑評估其分布，不以單次結果下定論。

**狀態隔離**：不同參數變體（策略名稱 + Top N + 停損 + 產業上限）各自擁有獨立的網路與訓練資料庫，同一程式行程內依序執行時互不污染。


## 階段 4：決策（挑一個門檻）

當可用樣本足夠（$\ge 200$）且特徵可算時，選「模型預測期望報酬最高」的動作；否則退回基準門檻：

$$a^* = \begin{cases}
\arg\max_a \ \text{網路預測報酬}(f)_a & \text{樣本充足且特徵可用} \\
(2.0,\ 0.0) \ \text{（基準門檻）} & \text{否則}
\end{cases}$$

其中 $\arg\max$（取最大者）意即「在 9 個門檻的預測報酬中，挑數值最高的那一個」。

動作選單：

$$\mathcal{A} = \{\text{SKIP}\} \cup \{(e, x) : e \in \{1.5, 2.0, 2.5, 3.0\},\ x \in \{0.0, 0.5\}\}, \qquad |\mathcal{A}| = 9$$

**三個結構性保證**：

1. 選單內含基準 $(2.0, 0.0)$ → 本策略的可選空間**涵蓋**純 Z-Score 狀態機；
2. 樣本不足時退回基準 → 累積初期的行為等同純 Z-Score；
3. SKIP 提供「拒絕交易」的選擇權 → 預期報酬為負的配對可整期空手，不被迫進場。


## 階段 5：反事實標籤生成（產生學習教材）

> **反事實（counterfactual）的白話**：即使本期實際只執行了一個門檻，我們仍對「沒選的其他門檻」在同一段交易期價格上**逐一模擬一遍**，算出它們各自會有的報酬。

無論本期實際選了哪個動作，都對**全部 8 個非 SKIP 門檻**在本期交易資料上逐一模擬（與正式交易同一套狀態機邏輯與費用會計），記錄各門檻的期末報酬率（%）作為 9 維標籤向量（SKIP 恆為 0）：

$$r_a = \frac{\text{損益}_a}{C_{\text{每配對}}} \times 100, \quad a \in \mathcal{A}$$

樣本 $(f,\ r,\ t_e)$（特徵、9 門檻報酬、結束日）存入緩衝區，供**之後**的期別訓練使用（本期不用，見階段 3 的資格規則，以確保無前視）。

**為何這讓問題變簡單**：因為 9 個門檻的報酬事後全部可觀察（全資訊），模型等於拿到「完整答案卷」，可直接以監督回歸學習「什麼特徵下、哪個門檻報酬最高」；不像一般強化學習只能看到自己選的那個結果（部分回饋），必須靠試誤探索。這也是本策略採用監督回歸、而非 Q-learning 等試誤方法的根本原因。


## 階段 6：以選定門檻執行正式模擬

選定 $(e, x)$ 後，整個交易期執行標準 Z-Score 狀態機：

| 條件 | 動作 |
| :--- | :--- |
| $Z_t > e$（空手時） | 空 A、多 B（風險中性配置 $v_A = C/W$、$v_B = |\beta|C/W$） |
| $Z_t < -e$ | 多 A、空 B |
| 空頭且 $Z_t \le x$／多頭且 $Z_t \ge -x$ | 平倉（可再進場） |
| 期末仍持倉 | `PERIOD_END_EXIT` 強制結算 |
| 選擇 SKIP | 整期 `HOLD_CASH (SKIP)` |

- 費用會計與 Z-Score 狀態機相同：進出場各扣 $(\text{fee} + \text{slippage}) \times$ 兩腳名目金額
- 最後一日不開新倉（$i < T-1$ 才允許進場）
- 輸出交易日誌欄位與 Z-Score 狀態機完全一致（`ZScore`、`Position`、`Status`、`Daily_Delta` 等）


# 參數總表

| 參數 | 值 | 對應環節 | 說明 |
| :--- | :---: | :--- | :--- |
| 動作選單 | 跳過 + 進場{1.5,2,2.5,3}×出場{0,0.5} | 決策 | 共 9 個候選門檻 |
| 基準（保底）門檻 | 偏離 2.0 進場、回歸 0 出場 | 決策 | 訓練樣本不足時的預設，等同 Z-Score |
| 神經網路寬度 | 兩層各 64 | 決策 | 由 12 特徵預測 9 門檻報酬 |
| 學習率 | 0.001 | 學習 | 參數更新步長 |
| 每次訓練輪數 | 40 | 學習 | 資料完整看過的遍數 |
| 啟用學習的最低樣本 | 200 | 學習/決策 | 未達則用保底門檻 |
| 特徵維度 | 12 | 特徵 | 形成期配對性質（見階段 2） |
| 隨機種子 | 不固定 | 學習 | 數值變異以多次獨立重跑評估 |
| 每配對資金 / 往返成本 | 10,000 / 0.58% | 執行 | 與 Z-Score 基準相同 |
